This is a credit card Fraud detection model in which I have to perrform challenges like handeling imbalanced data using SMOTE, Compare the perrformance of both the models i.e logistic regression and random forest classifier, and evaluate the model using accuracy, precision, recall and f1. 

Import the necessary libraries Pandas for Data loading, manipulation and cleaning
numpy for numerical operations and matplotlib and seaborn for visualizations

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [4]:
from imblearn.over_sampling import SMOTE

Loading the dataset with the help of pandas dataframes

In [5]:
df = pd.read_csv("credit_card_fraud_dataset.csv")
df.head()

,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0


In [6]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   TransactionID    100000 non-null  int64  
 1   TransactionDate  100000 non-null  object 
 2   Amount           100000 non-null  float64
 3   MerchantID       100000 non-null  int64  
 4   TransactionType  100000 non-null  object 
 5   Location         100000 non-null  object 
 6   IsFraud          100000 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 5.3+ MB


,TransactionID,Amount,MerchantID,IsFraud
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,50000.500000,2497.092666,501.676070,0.010000
std,28867.657797,1442.415999,288.715868,0.099499
min,1.000000,1.050000,1.000000,0.000000
25%,25000.750000,1247.955000,252.000000,0.000000
50%,50000.500000,2496.500000,503.000000,0.000000
75%,75000.250000,3743.592500,753.000000,0.000000
max,100000.000000,4999.770000,1000.000000,1.000000


In [7]:
df['IsFraud'].value_counts()

IsFraud
0    99000
1     1000
Name: count, dtype: int64

This step counts the fraud values i.e (1) for fraud and (0) for non fraud 
also tells us about the class imbalance which is the challenge in the fraud detection models

#Data Preprocessing

In [8]:
x = df.drop('IsFraud', axis=1)
y = df['IsFraud'] #Separate Features and target so that the model could understand the feature and the target output.

In [9]:
df.drop(['TransactionID'], axis = 1, inplace = True)

In this project transactionId is the unique identifier as usual but not required for fraud detection so the column transaction id is removed

In [10]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])

df['Hour']=df['TransactionDate'].dt.hour
df['Day']=df['TransactionDate'].dt.day
df['Weekday']=df['TransactionDate'].dt.weekday

df.drop(['TransactionDate'], axis = 1, inplace = True)

Converts the Transaction Date column into datetime format which was originally in string, this feature engineering is required to extract meaningful time features, it is necessary beacause sometimes frauds can be depend on the time patterns, and also improves the model performance, also droppedd the original datetime column so that it prevent data reddundancy.

In [11]:
df = pd.get_dummies(df, columns=['TransactionType', 'Location'],drop_first = True) # one hot-coding

This step converts the categorial data into binary (1/0) columns, drop_first = true avoids the dummy variable trap

In [12]:
x = df.drop('IsFraud', axis = 1)
y = df['IsFraud']

Here we are splitting the features and targets i.e: x= input features and y = target vakues(fraud or not) this is required in the supervised learning models.

In [13]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 42, stratify = y)

This step is essential for splitting the training and testing data where it splits training data into 80% and testing into 20%
    stratify keeps the fraud ratio same in both the training and testing sets

In [14]:
# Handle Imbalanced data with the help of SMOTE
smote = SMOTE(random_state = 42)
x_train_sm, y_train_sm = smote.fit_resample(x_train, y_train)


SMOTE is a Synthatic Minority over-sampling technique it is a method which is used to balance and imbalance datasets by creating synthatic data points for the monority class
here fraud data is highly imbalanced, SMOTE helps in creating synthetic fraud samples
also prevent model bias towards majority class

In [15]:
x_train.dtypes

Amount                    float64
MerchantID                  int64
Hour                        int32
Day                         int32
Weekday                     int32
TransactionType_refund       bool
Location_Dallas              bool
Location_Houston             bool
Location_Los Angeles         bool
Location_New York            bool
Location_Philadelphia        bool
Location_Phoenix             bool
Location_San Antonio         bool
Location_San Diego           bool
Location_San Jose            bool
dtype: object

In [16]:
print(y_train.value_counts())
print(y_train_sm.value_counts())

IsFraud
0    79200
1      800
Name: count, dtype: int64
IsFraud
0    79200
1    79200
Name: count, dtype: int64


this step is use to confirms tha balanced class distribution after using the SMOTE

In [17]:
# Feature Scaling
scaler = StandardScaler()
x_train_sm = scaler.fit_transform(x_train_sm)
x_test = scaler.transform(x_test) 

Standard Scalar is used here to standardize features (mean=0, std=1), which is required for the distance based models also it prevents dominance of large-scale features

In [18]:
#Train Model
#Logistic Regression
lr = LogisticRegression()
lr.fit(x_train_sm, y_train_sm)

y_pred_lr = lr.predict(x_test)

logistic regression is a baseline classification model. 
y_pred_lr is used for predicting fraud/ non fraud on unseen data

In [19]:
#Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train_sm, y_train_sm)
y_pred_rf = rf.predict(x_test)

Ensemble model = Multiple decision trees it handles non-linear relationships class_weight='balanced' penalizes fraud misclassification

In [20]:
#Model Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [21]:
def evaluate_model(y_test, y_pred, model_name):
    print(f"\n{model_name} Performance:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1_Score:", f1_score(y_test, y_pred))
    

In [22]:
from sklearn.metrics import confusion_matrix
print("Logistic Regression confusion matrix:")
print(confusion_matrix(y_test, y_pred_lr))

print("\nRandom Forest confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Logistic Regression confusion matrix:
[[19800     0]
 [  200     0]]

Random Forest confusion matrix:
[[19749    51]
 [  200     0]]


Confusion matrix is used to understand how a classification model is making prediction not just how many predictions are correct. 
it shows TP, FP, FN, TN which helps visually understand model bheavior this is the strong evaluation tool for fraud detection.

In [23]:
# Change decision threshold
y_prob_lr = lr.predict_proba(x_test)[:, 1]
#Threshold lower
y_pred_lr = (y_prob_lr >= 0.3).astype(int)
#evaluate the threshold value
evaluate_model(y_test, y_pred_lr, "LogisticRegression (threshold=0.3)")


LogisticRegression (threshold=0.3) Performance:
Accuracy: 0.7147
Precision: 0.007689556509298999
Recall: 0.215
F1_Score: 0.014848066298342542


Default logical threshold is 0.5 whic is too strict for this model so we have changed it to 0.3, lower threshold improves fraud detection.

In [24]:
#Use class weight 
lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(x_train_sm, y_train_sm)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators = 200,
    random_state = 42,
    class_weight = 'balanced'
)
rf.fit(x_train_sm, y_train_sm)
y_pred_rf = rf.predict(x_test)

In [ ]:
print(y_test.value_counts())

In [ ]:
from sklearn.metrics import roc_auc_score
print("ROC-AUC LR:", roc_auc_score(y_test, y_prob_lr))

We used ROC_AUC to evaluate the overall discriminative power of the Logistic regression model, especially because the dataset we have is imbalanced.

In [ ]:
evaluate_model(y_test, y_pred_lr, "LogisticRegression")
evaluate_model(y_test, y_pred_rf, "RandomForestClassifier")

With the help of model evaluation we observed the:
Accuracy ~0.98
precision, Recall, F1 = 0
which tells that the model predicted only non-fraud, Accuracy is misleading dur to imbalance this factors shows that why accuracy alone is sufficient.

In [ ]:
#model comaprision table
results = pd.DataFrame({
    "model":["Logistic Regression", "Random Forest Classifier"],
    "Accuracy":[
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Precision":[
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall":[
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score":[
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ]
})
results



Model Comparision Table Summarizes model performance in one place, this helps comparision clear to understand. \
Conclusion: Using SMOTE in this project was necessary, Accuracy alone is misleading and Random Forest with threshold tuning perform better.